# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hassaan-Raza/FlyRank-Intership/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, cross_val_predict, GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import os

df = pd.read_csv("content_refresh_anonymized.csv")
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

feature_cols = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'engaged_sessions_90d',
    'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions',
    'days_with_sessions', 'content_age_days', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct'
]

X = df[feature_cols].fillna(0)
y = df['is_declining']
groups = df['client_id']

# Honest scoring: out-of-fold predictions, every row scored by a model
# that never trained on that row (avoids the training-memorization
# artifact caught in review, where in-sample scoring produced fake 1.0s)
gkf = GroupKFold(n_splits=5)
rf_cv = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
oof_probs = cross_val_predict(rf_cv, X, y, groups=groups, cv=gkf, method='predict_proba')[:, 1]
df['risk_score'] = oof_probs

# Fit one final model on all data for feature importance / reporting only
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced').fit(X, y)

# Reason codes, same logic as Week 4/7 baseline
df['reason_code'] = 'monitor'
df.loc[(df['trend_direction'] == 'down') & (df['impressions_90d'] >= 100), 'reason_code'] = 'declining_with_demand'
df.loc[(df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500), 'reason_code'] = 'stale_visible_page'
df.loc[(df['avg_position'] <= 20) & (df['ctr'] < 0.5) & (df['impressions_90d'] >= 500), 'reason_code'] = 'low_ctr_visible_page'

# Descriptive archetypes (not fixed personas, snapshot-based)
archetype_features = df[['impressions_90d', 'avg_position', 'content_age_days', 'word_count']].fillna(0)
scaled = StandardScaler().fit_transform(archetype_features)
df['archetype'] = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(scaled)

# Mapping based on actual observed cluster profiles (checked, not guessed):
# 0: moderate impressions, strong position, older, moderate length -> established, watch for decay
# 1: low impressions, weak position, longest words -> underperforming despite length
# 2: moderate impressions, strong position, youngest, longest words -> new, still building
# 3: very high impressions, strong position, small group -> highest-value outliers
archetype_action_map = {
    0: 'protect_and_monitor',
    1: 'review_for_refresh',
    2: 'monitor',
    3: 'protect_priority'
}
df['recommended_action'] = df['archetype'].map(archetype_action_map)

ranked_queue = df.sort_values('risk_score', ascending=False)[
    ['content_id', 'risk_score', 'reason_code', 'recommended_action', 'archetype']
]

os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)
ranked_queue.to_csv('work/outputs/content_action_playbook.csv', index=False)

print(ranked_queue.head(20))
print(f"\nTotal ranked: {len(ranked_queue)}")
print(f"\nArchetype profile (mean values):")
print(df.groupby('archetype')[['impressions_90d', 'avg_position', 'content_age_days', 'word_count']].mean())

                 content_id  risk_score            reason_code  \
21882  content_2dbab51b83c9       0.985   low_ctr_visible_page   
17451  content_d015eb800625       0.980   low_ctr_visible_page   
23688  content_3cec1c2404d1       0.975                monitor   
7224   content_a53d1c4231b3       0.975   low_ctr_visible_page   
6332   content_dfc45d59cf17       0.975  declining_with_demand   
15817  content_f79387f83703       0.970   low_ctr_visible_page   
16186  content_fd2075ea5f6a       0.970  declining_with_demand   
13168  content_585bbd651cc4       0.970                monitor   
24688  content_1e0605b35117       0.970                monitor   
27035  content_9c128be31943       0.965                monitor   
22042  content_2ba626fea4d6       0.965                monitor   
16330  content_253e7769f591       0.965  declining_with_demand   
22932  content_da27e8e0c580       0.965  declining_with_demand   
12644  content_bdc3bb0fa949       0.960                monitor   
9750   con

**Reason codes:** declining_with_demand (trend down + real impressions),
stale_visible_page (unchanged 180+ days, still visible), low_ctr_visible_page
(ranks well, underclicks). Everything else falls to monitor.

**Archetype → action mapping (from real cluster profiles, not placeholder
labels):**  established/aging high performers get protect_and_monitor, low-
visibility long-form content gets review_for_refresh, young content still
building momentum gets monitor, and the small group of extreme outlier
pages gets protect_priority above all else.

**Note on scoring:** risk_score uses out-of-fold predictions (GroupKFold, 5
splits), not in-sample scoring, avoiding the inflated near-1.0 scores that
appear when a model scores rows it was trained on.

## 2. Intended use and limits

**Intended use:** this playbook is decision support for a human content reviewer
with limited time, it ranks candidates, it does not decide or act on its own.
Every recommendation should be read as "worth a look," not "confirmed problem."

**Limits:** the risk_score comes from a model validated at AUC~0.60-0.78
depending on split honesty (see Week 6 audit), modest, not strong, predictive
power. trend_direction remains a current-window proxy, not a verified future
outcome. Archetypes are descriptive clusters from this snapshot, not fixed
personas, they may not hold on future data. No FlyRank product-computed
score was used as a feature, so this output reflects only observable
signals, not any existing internal FlyRank logic.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

**Human review rule:** every action above a review threshold gets a human look
before any change is made. Nothing in this queue should trigger an automatic
edit, publish, or removal.

**No-go list (must NOT be automated):**
- Do not auto-publish, auto-prune, or auto-redirect any page based on
  risk_score alone.
- Do not treat a "declining_with_demand" flag as proof a refresh will help,
  it identifies a candidate worth checking, not a guaranteed fix.
- Do not run this queue on a new client without re-validating, the model
  was trained on a specific client mix, and Week 6 showed a meaningful AUC
  gap between naive and grouped validation, meaning cross-client
  generalization is not guaranteed.
- Do not treat archetype cluster membership as a stable long-term label,
  clusters were fit on one snapshot.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

**Monitoring signals:** track the class balance of trend_direction month to
month, if it shifts far from the ~54/46 split seen in this snapshot, the
model's calibration may no longer hold. Track the reason_code distribution
over time, a sudden spike in any one reason code may signal a data pipeline
change rather than a real content pattern shift.

**Retrain triggers:** retrain if validated AUC (on a fresh client-grouped
holdout) drops meaningfully below the range established here, or on a fixed
quarterly cadence regardless, since this is a lightweight model on a single
snapshot, not a continuously-updated production system.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [5]:
import json
import matplotlib.pyplot as plt

importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False).head(10)
plt.figure(figsize=(8, 5))
importances.plot(kind='barh')
plt.xlabel('Importance')
plt.title('Top 10 Feature Importances (Random Forest)')
plt.tight_layout()
plt.savefig('work/figures/feature_importance.png', dpi=150)
plt.close()

metrics = {
    "model": "RandomForestClassifier",
    "scoring_method": "out-of-fold via GroupKFold(5), avoids in-sample leakage",
    "validation_design": "client-grouped holdout (GroupShuffleSplit) for AUC/AP reporting",
    "n_total_scored": len(df),
    "n_flagged_declining_with_demand": int((df['reason_code'] == 'declining_with_demand').sum()),
    "top_feature": importances.index[0],
    "top_feature_importance": round(float(importances.iloc[0]), 3)
}
with open('work/outputs/playbook_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("Exported: work/outputs/content_action_playbook.csv")
print("Exported: work/figures/feature_importance.png")
print("Exported: work/outputs/playbook_metrics.json")
print(metrics)

Exported: work/outputs/content_action_playbook.csv
Exported: work/figures/feature_importance.png
Exported: work/outputs/playbook_metrics.json
{'model': 'RandomForestClassifier', 'scoring_method': 'out-of-fold via GroupKFold(5), avoids in-sample leakage', 'validation_design': 'client-grouped holdout (GroupShuffleSplit) for AUC/AP reporting', 'n_total_scored': 30000, 'n_flagged_declining_with_demand': 7026, 'top_feature': 'avg_position', 'top_feature_importance': 0.122}


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.